# **Importing the dataset**

## Digital Music

In [1]:
import pandas as pd

# Load a JSON file into a pandas DataFrame
df1 = pd.read_json('/content/Digital_Music_5.json', lines=True)

print(f"The DataFrame has {df1.shape[0]} rows and {df1.shape[1]} columns.")

FileNotFoundError: File /content/Digital_Music_5.json does not exist

In [ ]:
# Randomly sample 15,000 rows
df1_sampled = df1.sample(n=15000, random_state=42) # Using random_state for reproducibility

print(f"The sampled DataFrame has {df1_sampled.shape[0]} rows and {df1_sampled.shape[1]} columns.")
df1_sampled.head()

In [ ]:
# Count the distribution of 'overall' ratings
rating_distribution = df1_sampled['overall'].value_counts().sort_index()

print("Distribution of Overall Ratings:")
print(rating_distribution)

In [ ]:
# drop all columns except the id review text and overall
df1_sampled_2 = df1_sampled[['overall', 'reviewText']]
df1_sampled_2.head()

# Luxury Beauty

In [ ]:
import pandas as pd

# Load a JSON file into a pandas DataFrame
df2 = pd.read_json('/content/Luxury_Beauty_5.json', lines=True)

print(f"/nThe DataFrame has {df2.shape[0]} rows and {df2.shape[1]} columns.")

In [ ]:
# Randomly sample 15,000 rows
df2_sampled = df2.sample(n=15000, random_state=42) # Using random_state for reproducibility

print(f"The sampled DataFrame has {df2_sampled.shape[0]} rows and {df2_sampled.shape[1]} columns.")
df2_sampled.head()

In [ ]:
# Count the distribution of 'overall' ratings
rating_distribution2 = df2_sampled['overall'].value_counts().sort_index()

print("Distribution of Overall Ratings:")
print(rating_distribution2)

In [ ]:
# drop all columns except the id review text and overall
df2_sampled_2 = df2_sampled[['overall', 'reviewText']]
df2_sampled_2.head()

# Software

In [ ]:
import pandas as pd

# Load a JSON file into a pandas DataFrame
df3_sampled = pd.read_json('/content/Software_5.json', lines=True)

print(f"/nThe DataFrame has {df3_sampled.shape[0]} rows and {df3_sampled.shape[1]} columns.")

In [ ]:
df3_sampled.head()

In [ ]:
# Count the distribution of 'overall' ratings
rating_distribution3 = df3_sampled['overall'].value_counts().sort_index()

print("Distribution of Overall Ratings:")
print(rating_distribution3)

In [ ]:
# drop all columns except the id review text and overall
df3_sampled_2 = df3_sampled[['overall', 'reviewText']]
df3_sampled_2.head()

# **Concatenate All datasets**

In [ ]:
# combine df_sampled_2, df2_sampled_2 and df3_sampled_3

df = pd.concat([df1_sampled_2, df2_sampled_2, df3_sampled_2])
print(f"/nThe DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.")

# **Train / Validation / Test Split (70 / 15 / 15)**

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: separate features and labels
X = df["reviewText"]
y = df["overall"]

# Step 2: first split 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Step 3: split temp into validation and test (15% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"X_train length: {len(X_train)}")
print(f"X_val length: {len(X_val)}")
print(f"X_test length: {len(X_test)}")

# **Preprocessing Pipeline**

## Text Cleaning

Typical operations:

1. lowercase
2. remove punctuation
3. remove numbers (optional)
4. remove extra spaces



In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)      # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)     # keep only letters
    text = re.sub(r"\s+", " ", text).strip() # remove extra spaces
    return text

X_train = X_train.apply(clean_text)
X_val = X_val.apply(clean_text)
X_test = X_test.apply(clean_text)

## Tokenization

In [ ]:
def tokenize(text):
    return text.split()

X_train_tokens = X_train.apply(tokenize)
X_val_tokens = X_val.apply(tokenize)
X_test_tokens = X_test.apply(tokenize)

## Vocabulary
On training set only

In [ ]:
from collections import Counter

counter = Counter()

for tokens in X_train_tokens:
    counter.update(tokens)

vocab = {word: idx+2 for idx, (word, _) in enumerate(counter.items())}

MAX_VOCAB = 20000

# Keep only top MAX_VOCAB words by frequency
most_common = counter.most_common(MAX_VOCAB)
vocab = {word: idx+2 for idx, (word, _) in enumerate(most_common)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

print(f"Vocabulary size capped at: {len(vocab)}")

## Convert tokens into indices

In [ ]:
def encode(tokens):
    return [vocab.get(word, 1) for word in tokens]  # 1 = UNK

X_train_seq = X_train_tokens.apply(encode)
X_val_seq = X_val_tokens.apply(encode)
X_test_seq = X_test_tokens.apply(encode)

## Padding / Truncation

In [ ]:
MAX_LEN = 200

import numpy as np

def pad_sequence(seq):
    if len(seq) > MAX_LEN:
        return seq[:MAX_LEN]
    else:
        return seq + [0] * (MAX_LEN - len(seq))

X_train_pad = np.array(X_train_seq.apply(pad_sequence).tolist())
X_val_pad = np.array(X_val_seq.apply(pad_sequence).tolist())
X_test_pad = np.array(X_test_seq.apply(pad_sequence).tolist())

# **Part A:  Encoder Model for Understanding**

# Three-class formulation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score
import math

def map_sentiment(rating):
    if rating in [1, 2]:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

y_train_sentiment = y_train.apply(map_sentiment).values
y_val_sentiment   = y_val.apply(map_sentiment).values
y_test_sentiment  = y_test.apply(map_sentiment).values

print("Sentiment distribution (train):", np.unique(y_train_sentiment, return_counts=True))

# Derived feature (Intensity of review)

In [ ]:
STRONG_POSITIVE_WORDS = {
    "excellent", "outstanding", "exceptional", "superb", "fantastic",
    "incredible", "amazing", "wonderful", "brilliant", "perfect",
    "best", "greatest", "phenomenal", "extraordinary", "magnificent",
    "flawless", "absolutely", "definitely", "highly", "love"
}

STRONG_NEGATIVE_WORDS = {
    "terrible", "awful", "horrible", "dreadful", "disgusting",
    "worst", "useless", "pathetic", "atrocious", "abysmal",
    "never", "hate", "completely", "totally", "waste",
    "garbage", "trash", "broken", "disappointed", "dismal"
}

def map_intensity(text):
    tokens = set(str(text).lower().split())
    pos_count = len(tokens & STRONG_POSITIVE_WORDS)
    neg_count = len(tokens & STRONG_NEGATIVE_WORDS)
    if pos_count > neg_count:
        return 2  # Strong Positive
    elif neg_count > pos_count:
        return 0  # Strong Negative
    else:
        return 1  # Weak / Neutral

y_train_intensity = X_train.apply(map_intensity).values
y_val_intensity   = X_val.apply(map_intensity).values
y_test_intensity  = X_test.apply(map_intensity).values

print("Intensity distribution (train):", np.unique(y_train_intensity, return_counts=True))

# Initialise dataset using class

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, sentiment_labels, intensity_labels):
        self.sequences        = torch.tensor(sequences, dtype=torch.long)
        self.sentiment_labels = torch.tensor(sentiment_labels, dtype=torch.long)
        self.intensity_labels = torch.tensor(intensity_labels, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.sentiment_labels[idx], self.intensity_labels[idx]

BATCH_SIZE = 128

train_dataset = ReviewDataset(X_train_pad, y_train_sentiment, y_train_intensity)
val_dataset   = ReviewDataset(X_val_pad,   y_val_sentiment,   y_val_intensity)
test_dataset  = ReviewDataset(X_test_pad,  y_test_sentiment,  y_test_intensity)


# Loading Dataset

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
  # initalising of the sine and cosine wala part
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

  # adds the pe to the x
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Scaled Dot Product Attention

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        return torch.matmul(attn_weights, V), attn_weights

# Multi-Head Attention mechanism

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, x, mask=None):
        batch_size = x.size(0)
        Q = self.split_heads(self.W_Q(x), batch_size)
        K = self.split_heads(self.W_K(x), batch_size)
        V = self.split_heads(self.W_V(x), batch_size)
        attn_output, _ = self.attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.num_heads * self.d_k)
        return self.W_O(attn_output)

# Feed-Forward and Encoder Block

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads)
        self.ff      = FeedForward(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

# Full Encoder Model

In [ ]:
class ReviewEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2,
                 d_ff=256, max_len=200, dropout=0.1,
                 num_sentiment_classes=3, num_intensity_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.layers    = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.sentiment_head = nn.Linear(d_model, num_sentiment_classes)
        self.intensity_head = nn.Linear(d_model, num_intensity_classes)

    def forward(self, x):
        mask = (x != 0).unsqueeze(1).unsqueeze(2)
        emb  = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        emb  = self.pos_enc(emb)
        out  = emb
        for layer in self.layers:
            out = layer(out, mask)
        out = self.norm(out)

        # Mask-aware mean pooling -> review embedding
        lengths = mask.squeeze(1).squeeze(1).sum(dim=1, keepdim=True).float()
        pooled  = (out * mask.squeeze(1).transpose(1, 2)).sum(dim=1) / lengths

        return self.sentiment_head(pooled), self.intensity_head(pooled), pooled

# Run model Function

In [ ]:
def run_epoch(loader, model, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_sent_preds, all_sent_true = [], []
    all_int_preds,  all_int_true  = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for seqs, sent_labels, int_labels in loader:
            seqs, sent_labels, int_labels = (
                seqs.to(device), sent_labels.to(device), int_labels.to(device)
            )
            sent_logits, int_logits, _ = model(seqs)
            loss = criterion_sentiment(sent_logits, sent_labels) + \
                   criterion_intensity(int_logits, int_labels)

            if train and optimizer:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * seqs.size(0)
            all_sent_preds.extend(sent_logits.argmax(1).cpu().numpy())
            all_sent_true.extend(sent_labels.cpu().numpy())
            all_int_preds.extend(int_logits.argmax(1).cpu().numpy())
            all_int_true.extend(int_labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    sent_acc = accuracy_score(all_sent_true, all_sent_preds)
    int_acc  = accuracy_score(all_int_true,  all_int_preds)
    return avg_loss, sent_acc, int_acc

# Initiliase Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB_SIZE = len(vocab)
D_MODEL    = 64
NUM_HEADS  = 4
NUM_LAYERS = 2
D_FF       = 128
DROPOUT    = 0.1

model = ReviewEncoder(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS, d_ff=D_FF, max_len=MAX_LEN, dropout=DROPOUT
).to(device)

print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

 # Loss, Optimizer & Scheduler

In [ ]:
criterion_sentiment = nn.CrossEntropyLoss()
criterion_intensity = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Training

In [ ]:
EPOCHS = 10
history = {
    'train_loss': [], 'val_loss': [],
    'train_sent_acc': [], 'val_sent_acc': [],
    'train_int_acc':  [], 'val_int_acc':  []
}
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_sent, tr_int = run_epoch(train_loader, model, optimizer, train=True)
    vl_loss, vl_sent, vl_int = run_epoch(val_loader,   model, train=False)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss);     history['val_loss'].append(vl_loss)
    history['train_sent_acc'].append(tr_sent); history['val_sent_acc'].append(vl_sent)
    history['train_int_acc'].append(tr_int);   history['val_int_acc'].append(vl_int)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Loss: {tr_loss:.4f}/{vl_loss:.4f} | "
          f"Sent Acc: {tr_sent:.4f}/{vl_sent:.4f} | "
          f"Intensity Acc: {tr_int:.4f}/{vl_int:.4f}")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), 'best_encoder.pt')
        print("  Best model saved.")

In [ ]:
print(torch.cuda.is_available())
print(next(model.parameters()).device)